## Data Setup

In [1]:
%pip install pandas numpy matplotlib scikit-learn statsmodels
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_val_score, cross_val_predict, GridSearchCV, validation_curve)
from sklearn.linear_model import (LinearRegression, LogisticRegression, LogisticRegressionCV,
                                  RidgeCV, LassoCV, ElasticNetCV)
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import LinearSVC, SVC, SVR
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             log_loss, mean_squared_error, r2_score, roc_auc_score, roc_curve,
                             RocCurveDisplay)
from statsmodels.stats.outliers_influence import variance_inflation_factor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 25.6 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 30.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 30.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 29.3 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 35.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 28.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 19.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 30.9 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [statsmodels] [statsmodels]]a]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated p

In [2]:
raw = pd.read_csv("aei_claude_ai_2026-06-26.csv", dtype={"node_external_id": str})
raw["category_name"].value_counts()

FileNotFoundError: [Errno 2] No such file or directory: 'aei_claude_ai_2026-06-26.csv'

In [ ]:
keep = raw[(raw["geo_level"] == "global") &
           (raw["hierarchy_level"] == 0) &
           (raw["date_start"] == "2026-05-01") &
           (raw["category_name"].isin(["onet", "soc_occupation"]))]

print("Rows kept:", len(keep), "of", len(raw))
keep.head()

In [ ]:
def make_table(category):
    part = keep[keep["category_name"] == category]
    wide = part.pivot_table(index=["node_external_id", "node_name"],
                            columns="metric_id", values="value").reset_index()
    return wide

tasks = make_table("onet")                # one row per work task

print("Tasks:", tasks.shape)
tasks.head(3)

In [ ]:
def add_target(df):
    df = df.dropna(subset=["human_only_time_mean", "human_with_ai_time_mean"]).copy()
    df = df[df["human_with_ai_time_mean"] > 0]
    df["time_savings"] = (df["human_only_time_mean"] * 60) / df["human_with_ai_time_mean"]
    df["log_time_savings"] = np.log(df["time_savings"])
    return df

tasks = add_target(tasks)

tasks["log_time_savings"] = np.log(tasks["time_savings"])
tasks = add_target(tasks)

In [ ]:
# Dropping irrelevant columns for predicting time_savings from dataset (i.e., id fields)

tasks_cut = tasks.drop(columns=["time_savings", "node_external_id", "node_name", "human_only_time_mean", "human_with_ai_time_mean", "pct",
"collaboration_bucket_augmentation_pct", "collaboration_bucket_automation_pct", "collaboration_none_pct", "use_case_personal_pct", "artifact_none_pct"])

print(tasks_cut.shape) 

In [ ]:
# Remove rows with any blank values just for simplicity. There's only 6.

tasks_cut = tasks_cut.dropna().copy()
print(tasks_cut.shape)

In [ ]:
# Creating our X and y

y = tasks_cut["log_time_savings"]  
X = tasks_cut.drop(columns=["log_time_savings"])

In [ ]:
# Adding the best interaction and squared terms from Milestone 1
X_train["autonomy_edu"] = X_train["ai_autonomy_mean"] * X_train["ai_education_years_mean"]
X_train["autonomy_sq"] = X_train["ai_autonomy_mean"] ** 2

X_test["autonomy_edu"] = X_test["ai_autonomy_mean"] * X_test["ai_education_years_mean"]
X_test["autonomy_sq"] = X_test["ai_autonomy_mean"] ** 2

# OLS model with the added terms (predicting log time savings)
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test, has_constant="add")
model = sm.OLS(y_train, X_train_sm).fit()
pred = model.predict(X_test_sm)

print(f"OLS + added terms | Train R2: {model.rsquared:.3f}   Adj R2: {model.rsquared_adj:.3f}")
print(f"OLS + added terms | Test R2: {r2_score(y_test, pred):.3f}   "
      f"Test RMSE: {np.sqrt(mean_squared_error(y_test, pred)):.3f} (log units)")

# Ranking statistically significant coefficients (t value)
coef = pd.DataFrame({"coef": model.params, "p_value": model.pvalues,
                     "abs_t": model.tvalues.abs()}).drop("const")
print(coef.sort_values("abs_t", ascending=False).head(10).round(3))


# Scaling the features (needed for the models below)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# One function to print the same scores for every model
def score(model, name, X_tr, X_te, n_features):
    train_r2 = r2_score(y_train, model.predict(X_tr))
    n = len(y_train)
    adj_r2 = 1 - (1 - train_r2) * (n - 1) / (n - n_features - 1)
    pred = model.predict(X_te)
    print(name, "| Train R2:", round(train_r2, 3), "| Adj R2:", round(adj_r2, 3),
          "| Test R2:", round(r2_score(y_test, pred), 3),
          "| Test RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 3))


# Ridge, Lasso, Elastic Net
alphas = np.logspace(-5, 2, 50)   # penalty strengths from 0.00001 to 100

ridge = RidgeCV(alphas=alphas, cv=5).fit(X_train_s, y_train)
lasso = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000).fit(X_train_s, y_train)
enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], alphas=alphas, cv=5,
                    random_state=42, max_iter=10000).fit(X_train_s, y_train)

print("Best alpha - Ridge:", ridge.alpha_)
print("Best alpha - Lasso:", lasso.alpha_)
print("Best alpha - Elastic Net:", enet.alpha_, "| l1_ratio:", enet.l1_ratio_)

p = X_train_s.shape[1]   # number of features (45)
score(ridge, "Ridge", X_train_s, X_test_s, p)
score(lasso, "Lasso", X_train_s, X_test_s, p)
score(enet, "Elastic Net", X_train_s, X_test_s, p)


# Forward selection (10 features)
selector = SequentialFeatureSelector(LinearRegression(), n_features_to_select=10,
                                     direction="forward", cv=5)
selector.fit(X_train_s, y_train)
keep = selector.get_support()
print("Forward selection picked:", list(X_train.columns[keep]))

fwd_model = LinearRegression().fit(X_train_s[:, keep], y_train)
score(fwd_model, "Forward (10)", X_train_s[:, keep], X_test_s[:, keep], 10)


# PCR and PLSR for different numbers of components
for n in [5, 10, 20, 30]:
    pca_n = PCA(n_components=n)
    X_train_pca = pca_n.fit_transform(X_train_s)
    X_test_pca = pca_n.transform(X_test_s)
    pcr_model = LinearRegression().fit(X_train_pca, y_train)
    score(pcr_model, f"PCR ({n})", X_train_pca, X_test_pca, n)

    pls = PLSRegression(n_components=n).fit(X_train_s, y_train)
    score(pls, f"PLSR ({n})", X_train_s, X_test_s, n)


# Tuned RBF SVR
rbf_grid = GridSearchCV(SVR(kernel="rbf"),
                        param_grid={"C": [0.1, 1, 10, 100], "gamma": [0.001, 0.01, "scale"]},
                        scoring="r2", cv=5)
rbf_grid.fit(X_train_s, y_train)
print("Best RBF settings:", rbf_grid.best_params_)
score(rbf_grid, "Tuned RBF SVR", X_train_s, X_test_s, p)
